# 03 Homework 04 ETM — Graph Representation for Spatial Analysis

**Task 3 of the Final Assignment**

This notebook converts the four-floor building from the Modified Swiss Dwellings dataset into spatial graphs using TopologicPy. Each floor plan outline is loaded, sliced into a grid-based shell, and analysed with standard space-syntax metrics:

- Shortest path / navigation
- Closeness centrality (global integration)
- Betweenness centrality (choice)
- Community detection
- Degree centrality


## 1. Import the needed libraries

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color
import math
import time

## 2. Check the TopologicPy version

In [ ]:
print("This notebook requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

## 3. Set your renderer
* Visual Studio Code: `"vscode"`
* Google Colab: `"colab"`
* Browser: `"browser"`

In [ ]:
renderer = "vscode"

## 4. Utility functions

In [ ]:
from collections import defaultdict

def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for f in faces:
        d = Topology.Dictionary(f)
        for key in Dictionary.Keys(d):
            if key != "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts.get(str(value), None)
            if f:
                f = Topology.SetDictionary(f, d)

def _trace_boundary_loops(all_mesh_faces):
    """Return all boundary loops of a flat triangulated mesh as Wire objects.

    Boundary edges are those shared by exactly one face. The loops are returned
    sorted by enclosed area, largest first, so loops[0] is always the outer perimeter.
    """
    tol = 4
    def vk(v):
        return tuple(round(c, tol) for c in Vertex.Coordinates(v))

    cnt = defaultdict(int)
    edge_vks = {}
    for face in all_mesh_faces:
        for e in (Topology.Edges(face) or []):
            vs = Topology.Vertices(e)
            k = tuple(sorted([vk(vs[0]), vk(vs[1])]))
            cnt[k] += 1
            edge_vks[k] = (vk(vs[0]), vk(vs[1]))

    b_keys = {k for k, c in cnt.items() if c == 1}
    adj = defaultdict(list)
    for k in b_keys:
        k0, k1 = edge_vks[k]
        adj[k0].append(k1)
        adj[k1].append(k0)

    visited, loops = set(), []
    for start in list(adj.keys()):
        if start in visited:
            continue
        loop, curr = [start], start
        visited.add(start)
        while True:
            nxt = next((n for n in adj[curr] if n not in visited), None)
            if nxt is None:
                break
            visited.add(nxt)
            loop.append(nxt)
            curr = nxt
        if len(loop) > 2:
            loops.append(loop)

    def loop_area(loop):
        tverts = [Vertex.ByCoordinates(*c) for c in loop]
        w = Wire.ByVertices(tverts, close=True)
        f = Face.ByWire(w) if w else None
        return Face.Area(f) if f else 0

    loops.sort(key=loop_area, reverse=True)  # largest area first = outer perimeter first
    wires = []
    for loop in loops:
        tverts = [Vertex.ByCoordinates(*c) for c in loop]
        w = Wire.ByVertices(tverts, close=True)
        if w:
            wires.append(w)
    return wires


def load_floor_face(obj_path):
    """Load a Rhino OBJ surface export and return the floor plan as a Face.

    N-gon OBJ (Ground Floor, Level 3): the single polygon is returned directly.
    Mesh OBJ (Level 1, Level 2): boundary loops are traced and sorted by enclosed
    area.  The outer perimeter becomes the external boundary; any interior voids
    (stair shafts) are passed to Face.ByWires so the face correctly shows all walls.
    """
    objects = Topology.ByOBJPath(obj_path)
    if not isinstance(objects, list):
        objects = [objects]

    all_faces = []
    for obj in objects:
        faces = Topology.Faces(obj)
        if faces:
            all_faces.extend(faces)

    if not all_faces:
        return None

    # N-gon case: one large polygon IS the floor plan outline
    all_faces.sort(key=lambda f: len(Topology.Vertices(f) or []), reverse=True)
    best_verts = len(Topology.Vertices(all_faces[0]) or [])
    if best_verts > 4:
        return all_faces[0]

    # Mesh case: trace all boundary loops, sorted largest-area-first
    wires = _trace_boundary_loops(all_mesh_faces=all_faces)
    if not wires:
        return None

    outer_wire  = wires[0]
    inner_wires = wires[1:]

    if inner_wires:
        face = Face.ByWires(outer_wire, inner_wires)
        if face:
            return face

    return Face.ByWire(outer_wire)


## 5. Load floor plan outlines

One OBJ file per floor, each containing the 2D outline of that level.

In [ ]:
BASE_FP = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\FloorPlans"

FLOOR_PATHS = {
    "Ground Floor": BASE_FP + r"\GroungFloor.obj",
    "Level 1":      BASE_FP + r"\Level1.obj",
    "Level 2":      BASE_FP + r"\Level2.obj",
    "Level 3":      BASE_FP + r"\Level3.obj",
}

floor_faces  = {}   # Face per floor
floor_bounds = {}   # Bounding box per floor

for name, path in FLOOR_PATHS.items():
    print(f"Loading {name}...")
    face = load_floor_face(path)
    if face:
        floor_faces[name] = face
        n_verts = len(Topology.Vertices(face) or [])
        b_r = Wire.BoundingRectangle(face)
        d   = Topology.Dictionary(b_r)
        bounds = {k: Dictionary.ValueAtKey(d, k)
                  for k in ["xmin", "xmax", "ymin", "ymax", "width", "length"]}
        floor_bounds[name] = bounds
        print(f"  OK — {n_verts} boundary vertices, "
              f"{bounds['width']:.2f} x {bounds['length']:.2f} m  "
              f"(UV extents: u=[{bounds['xmin']:.2f},{bounds['xmax']:.2f}]  "
              f"v=[{bounds['ymin']:.2f},{bounds['ymax']:.2f}])")
    else:
        print(f"  FAILED — could not load face from {path}")

## 6. Visualize floor plan outlines

Each floor plan is shown as a 2D face — the outline that will be sliced into a spatial graph.

In [ ]:
for name, face in floor_faces.items():
    print(f"\n── {name} ──")
    Topology.Show(
        face,
        camera=[0, 0, 6],
        faceColor=[210, 210, 250],
        faceOpacity=1,
        edgeColor="white",
        edgeWidth=2,
        showVertices=False,
        backgroundColor="black",
        width=800, height=500,
        renderer=renderer
    )

## 7. Create grid overlays and slice into shells

Each floor plan is sliced with a 1 m × 1 m regular grid. Every grid cell becomes a node in the analysis graph — this is the spatial graph representation used for the topological metrics below.

Each face is tagged with a unique `face_id` so that analysis results can be transferred back for visualisation.

In [ ]:
GRID_STEP = 1

floor_shells  = {}  # Shell per floor
floor_f_lists = {}  # List of face objects per floor

for name, face in floor_faces.items():
    b = floor_bounds[name]
    w = b["width"]
    l = b["length"]

    # The grid origin is Face.VertexByParameters(face, 0,0) — one corner of the face.
    # We don't know which corner, so extend the range symmetrically in both directions
    # by the full face size. clip=True removes everything outside the face boundary,
    # guaranteeing every square metre of the floor plan is covered.
    u_ext = math.ceil(w) + GRID_STEP
    v_ext = math.ceil(l) + GRID_STEP
    uRange = list(range(-u_ext, u_ext + 1, GRID_STEP))
    vRange = list(range(-v_ext, v_ext + 1, GRID_STEP))

    grid  = Grid.EdgesByDistances(face, clip=True, uRange=uRange, vRange=vRange)
    shell = Topology.Slice(face, grid)
    f_list = Topology.Faces(shell)
    for i, f in enumerate(f_list):
        f = Topology.SetDictionary(f, Dictionary.ByKeyValue("face_id", f"face_{i+1}"))
    floor_shells[name]  = shell
    floor_f_lists[name] = f_list
    print(f"{name}: {len(f_list)} grid cells  (grid extent ±{u_ext} × ±{v_ext})")

## 8. Show the grid shells

In [ ]:
for name, shell in floor_shells.items():
    print(f"\n── {name} ──")
    Topology.Show(
        shell,
        camera=[0, 0, 6],
        faceColor=[210, 210, 250], faceOpacity=0.9,
        edgeColor="black", edgeWidth=1,
        showVertices=False,
        backgroundColor="black",
        width=800, height=500,
        renderer=renderer
    )

## 9. Build navigation and analysis graphs

Two graphs are derived from each shell:

- **Analysis graph** (`Graph.ByTopology`) — centroid-based dual graph; one node per grid cell, edges connect adjacent cells. Used for centrality metrics.
- **Navigation graph** (`viaSharedTopologies=True`) — connectivity graph suitable for routing (shortest path).

In [ ]:
floor_g_analysis   = {}  # Analysis graph per floor
floor_g_navigation = {}  # Navigation graph per floor
floor_g_verts      = {}  # Graph vertices per floor

for name, shell in floor_shells.items():
    g_nav = Graph.ByTopology(shell, direct=False, viaSharedTopologies=True)
    g_ana = Graph.ByTopology(shell)
    verts = Graph.Vertices(g_ana)
    floor_g_navigation[name] = g_nav
    floor_g_analysis[name]   = g_ana
    floor_g_verts[name]      = verts
    print(f"{name}: {len(verts)} nodes, {len(Graph.Edges(g_ana))} edges")

## 10. Show the analysis graphs

In [ ]:
for name in floor_faces:
    print(f"\n── {name} ──")
    Topology.Show(
        floor_g_analysis[name],
        camera=[0, 0, 6],
        vertexSize=4, vertexColor="red",
        edgeColor="lightgrey",
        backgroundColor="black",
        width=800, height=500,
        renderer=renderer
    )

## 11. Shortest Path

The shortest path is computed between the upper-left and lower-right corners of each floor using the navigation graph. The direct path is shown in red; the geometrically straightened path (fewest turns, still inside the floor plan) is shown in blue.

This reveals the primary movement spine of each floor.

In [ ]:
for name, face in floor_faces.items():
    b = floor_bounds[name]
    start_v = Vertex.ByCoordinates(b["xmin"] + 2, b["ymax"] - 2, 0)
    end_v   = Vertex.ByCoordinates(b["xmax"] - 2, b["ymin"] + 2, 0)
    crg = Graph.CompiledRoutingGraph(floor_g_navigation[name], precomputeTurns=False)

    t0 = time.time()
    sp = Graph.ShortestPath(crg, vertexA=start_v, vertexB=end_v)
    print(f"{name}: path length = {Wire.Length(sp):.2f} m  ({time.time()-t0:.2f}s)")

    straight = Wire.Straighten(sp, host=face)
    print(f"  straightened = {Wire.Length(straight):.2f} m")

    for e in (Topology.Edges(sp) or []):
        Topology.SetDictionary(e, Dictionary.ByKeysValues(["width","color"],[6,"red"]))
    for e in (Topology.Edges(straight) or []):
        Topology.SetDictionary(e, Dictionary.ByKeysValues(["width","color"],[6,"blue"]))

    print(f"\n── {name} ──")
    Topology.Show(
        face, sp, straight,
        camera=[0, 0, 6],
        faceColor=[210, 210, 250], faceOpacity=1,
        edgeColorKey="color", edgeWidthKey="width",
        backgroundColor="black",
        width=800, height=500,
        renderer=renderer
    )

## 12. Closeness Centrality — Global Integration

Closeness centrality measures how easily a space can be reached from all other spaces (reciprocal of the mean shortest-path distance). In space syntax this corresponds to **global integration** — high values indicate spaces that are the most accessible across the entire floor plan.

Warm colours (yellow → red) = high integration. Cool colours (blue → purple) = low integration.

In [ ]:
for name in floor_faces:
    print(f"\n── {name} ──")
    _ = Graph.ClosenessCentrality(floor_g_analysis[name], colorScale="thermal")
    g_verts = floor_g_verts[name]
    reset_dictionaries(floor_shells[name])
    f_list = Topology.Faces(floor_shells[name])
    _ = transfer_dicts_by_key(f_list, g_verts, "face_id")
    Topology.Show(
        f_list,
        faceColorKey="cc_color", faceOpacity=1,
        showEdges=False, showVertices=False,
        camera=[0, 0, 6],
        backgroundColor="black",
        width=800, height=500,
        renderer=renderer
    )

## 13. Betweenness Centrality — Choice

Betweenness centrality counts how often a cell lies on the shortest path between any two other cells. In space syntax this corresponds to **choice** — high-betweenness cells are likely movement corridors that experience the most through-traffic.

Warm colours = high choice (likely circulation routes). Cool colours = low choice (dead-end or private spaces).

In [ ]:
for name in floor_faces:
    print(f"\n── {name} ──")
    _ = Graph.BetweennessCentrality(floor_g_analysis[name], normalize=True, colorScale="thermal")
    g_verts = floor_g_verts[name]
    reset_dictionaries(floor_shells[name])
    f_list = Topology.Faces(floor_shells[name])
    _ = transfer_dicts_by_key(f_list, g_verts, "face_id")
    Topology.Show(
        f_list,
        faceColorKey="bc_color", faceOpacity=1,
        showEdges=False, showVertices=False,
        camera=[0, 0, 6],
        backgroundColor="black",
        width=800, height=500,
        renderer=renderer
    )

## 14. Community Detection

Community detection (Louvain method) partitions the graph into spatial clusters — groups of grid cells that are more strongly connected to each other than to the rest of the floor plan. Each community typically corresponds to a coherent spatial zone (a room, a wing, or a circulation area).

Each detected community is then dissolved into a single face representing its spatial extent.

In [ ]:
floor_communities  = {}  # community face groups per floor
floor_com_graphs   = {}  # community-level graph per floor
floor_com_verts    = {}  # community-level graph vertices

for name in floor_faces:
    print(f"\n── {name} — community detection (may take a few minutes) ──")
    _ = Graph.CommunityPartition(floor_g_analysis[name], colorScale="thermal")
    g_verts = floor_g_verts[name]
    reset_dictionaries(floor_shells[name])
    f_list = Topology.Faces(floor_shells[name])
    _ = transfer_dicts_by_key(f_list, g_verts, "face_id")

    Topology.Show(
        f_list,
        faceColorKey="cp_color", faceOpacity=1,
        showEdges=False, showVertices=False,
        camera=[0, 0, 6],
        backgroundColor="black",
        width=800, height=500,
        renderer=renderer
    )

    # Dissolve each community into a single face
    bins = Topology.BinByDictionaryKey(f_list, key="community")
    bin_dict = bins[0]
    face_groups = []
    for key in list(bin_dict.keys()):
        bin_faces = bin_dict[key]
        temp_shell = Shell.ByFaces(bin_faces)
        if temp_shell is None:
            continue
        eb = Shell.ExternalBoundary(temp_shell)
        if eb is None:
            continue
        eb = Wire.RemoveCollinearEdges(eb)
        eb_face = Face.ByWire(eb)
        if eb_face:
            face_groups.append(eb_face)
    floor_communities[name] = face_groups
    print(f"  {len(face_groups)} communities detected")

    new_shell = Shell.ByFaces(face_groups)
    if new_shell:
        new_graph = Graph.ByTopology(new_shell)
        new_verts = Graph.Vertices(new_graph)
        for v in new_verts:
            v = Topology.SetDictionary(v, Dictionary.ByKeysValues(["color","size"],["red",12]))
        floor_com_graphs[name] = new_graph
        floor_com_verts[name]  = new_verts

        Topology.Show(
            new_shell, new_graph,
            faceOpacity=0.9,
            showEdges=True, showVertices=True,
            vertexSizeKey="size", vertexColorKey="color",
            camera=[0, 0, 6],
            backgroundColor="black",
            width=800, height=500,
            renderer=renderer
        )

## 15. Degree Centrality

Degree centrality is computed on the community-level graph (one node per community). It counts how many other communities each community directly borders. This highlights the most **connected zones** in the building — spaces that act as hubs between multiple areas.

The result is interpolated back to the original fine-grid vertices for spatial visualisation.

In [ ]:
for name in floor_faces:
    if name not in floor_com_graphs:
        print(f"{name}: no community graph available — skipping")
        continue
    print(f"\n── {name} ──")
    new_graph = floor_com_graphs[name]
    new_verts = floor_com_verts[name]
    g_verts   = floor_g_verts[name]

    degree_centralities = Graph.DegreeCentrality(new_graph, normalize=False)

    # Interpolate from community nodes to dense grid vertices
    for v in g_verts:
        Vertex.InterpolateValue(v, vertices=new_verts, n=3, key="degree_centrality")

    min_dc = min(degree_centralities)
    max_dc = max(degree_centralities)
    for v in g_verts:
        d   = Topology.Dictionary(v)
        dc  = Dictionary.ValueAtKey(d, "degree_centrality")
        col = Color.AnyToHex(Color.ByValueInRange(
            dc, minValue=min_dc, maxValue=max_dc, colorScale="thermal"))
        d = Dictionary.SetValueAtKey(d, "dc_color", col)
        v = Topology.SetDictionary(v, d)

    reset_dictionaries(floor_shells[name])
    f_list = Topology.Faces(floor_shells[name])
    _ = transfer_dicts_by_key(f_list, g_verts, "face_id")

    Topology.Show(
        f_list,
        faceColorKey="dc_color", faceOpacity=1,
        showEdges=False, showVertices=False,
        camera=[0, 0, 6],
        backgroundColor="black",
        width=800, height=500,
        renderer=renderer
    )